In [ ]:
# Cell 1: Install dependencies and mount Drive
!pip install timm huggingface_hub openslide-python h5py opencv-python-headless
!apt-get install -y openslide-tools

from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/hb-1968/prame-predict.git

In [ ]:
# Cell 2: HuggingFace login
from huggingface_hub import login
login()

In [ ]:
# Cell 3: Feature extraction pipeline
import requests
import shutil
import numpy as np
import pandas as pd
import torch
import timm
import h5py
import openslide
from pathlib import Path
from tqdm import tqdm
from importlib.machinery import SourceFileLoader
from concurrent.futures import ThreadPoolExecutor

# â”€â”€ Import tiling module from cloned repo â”€â”€
tile_module = SourceFileLoader("tile", "prame-predict/02_tile_wsi.py").load_module()
tile_slide = tile_module.tile_slide
_close_thread_handles = tile_module._close_thread_handles

# â”€â”€ Load UNI model â”€â”€
print("Loading UNI...")
model = timm.create_model(
    "vit_large_patch16_224",
    init_values=1e-5,
    num_classes=0,
    pretrained=False,
)

from huggingface_hub import hf_hub_download
ckpt_path = hf_hub_download(repo_id="MahmoodLab/uni", filename="pytorch_model.bin")
state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict, strict=True)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Running on: {device}")


def preprocess_batch(batch_np):
    """
    Vectorized preprocessing: uint8 numpy (N,224,224,3) -> normalized tensor.
    Both UNI and CONCH use mean=0.5, std=0.5, input_size=224x224.
    """
    batch = torch.from_numpy(batch_np).permute(0, 3, 1, 2).float().div_(255.0)
    batch.mul_(2.0).sub_(1.0)
    return batch.to(device, non_blocking=True)


# â”€â”€ Config â”€â”€
manifest = pd.read_csv("prame-predict/data/expression/slide_manifest.csv")
drive_emb = Path("/content/drive/MyDrive/prame-predict/embeddings/uni")
drive_emb.mkdir(parents=True, exist_ok=True)
local_wsi = Path("/content/wsi_batch")
local_wsi.mkdir(exist_ok=True)
local_tiles = Path("/content/tiles_tmp")
local_tiles.mkdir(exist_ok=True)

BATCH_SIZE_GPU = 128     # patches per forward pass (float16 halves VRAM)
MAX_PATCHES = 80000      # random sample cap per slide
DOWNLOAD_BATCH = 75      # slides per download batch


def download_slide(row, max_retries=3):
    """Download a single slide from GDC with retry logic."""
    local_path = local_wsi / row["file_name"]
    if local_path.exists():
        return local_path

    for attempt in range(max_retries):
        try:
            url = f"https://api.gdc.cancer.gov/data/{row['file_id']}"
            response = requests.get(url, stream=True, timeout=300)
            response.raise_for_status()
            with open(local_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=65536):
                    f.write(chunk)
            return local_path
        except Exception as e:
            print(f"  Retry {attempt + 1}/{max_retries} for {row['file_name']}: {e}")
            if local_path.exists():
                local_path.unlink()

    print(f"  FAILED after {max_retries} attempts: {row['file_name']}")
    return None


def process_slide(slide_path, slide_name):
    """Tile one slide, extract features with float16 inference, cleanup tiles."""
    slide_out = local_tiles / slide_name.replace(".svs", "")
    slide_out.mkdir(parents=True, exist_ok=True)

    # Tile with memmap output and max-patches cap
    num_patches, coords = tile_slide(
        slide_path, slide_out, workers=4, max_patches=MAX_PATCHES
    )
    np.save(slide_out / "coords.npy", np.array(coords))
    _close_thread_handles()

    if num_patches == 0:
        shutil.rmtree(slide_out)
        return None

    # Load patches via mmap (bounded RAM)
    patches = np.load(slide_out / "patches.npy", mmap_mode='r')

    features = []
    with torch.no_grad():
        for j in tqdm(range(0, len(patches), BATCH_SIZE_GPU),
                      desc="  Extracting", leave=False):
            batch_np = np.array(patches[j:j + BATCH_SIZE_GPU])
            batch = preprocess_batch(batch_np)

            with torch.amp.autocast("cuda"):
                feats = model(batch)

            features.append(feats.cpu().numpy().astype(np.float16))

    del patches

    # Cleanup tiles immediately
    shutil.rmtree(slide_out)

    return np.vstack(features), np.array(coords)


# â”€â”€ Filter to unprocessed slides â”€â”€
remaining = []
for _, row in manifest.iterrows():
    emb_path = drive_emb / row["file_name"].replace(".svs", ".h5")
    if not emb_path.exists():
        remaining.append(row)

remaining = pd.DataFrame(remaining)
print(f"Slides remaining: {len(remaining)} / {len(manifest)}")

# â”€â”€ Process in batches â”€â”€
for batch_start in range(0, len(remaining), DOWNLOAD_BATCH):
    batch = remaining.iloc[batch_start:batch_start + DOWNLOAD_BATCH]
    batch_end = min(batch_start + DOWNLOAD_BATCH, len(remaining))
    print(f"\n{'='*60}")
    print(f"BATCH {batch_start // DOWNLOAD_BATCH + 1}: "
          f"slides {batch_start + 1}-{batch_end} of {len(remaining)}")
    print(f"{'='*60}")

    # Download batch in parallel (8 threads)
    print(f"Downloading {len(batch)} slides...")
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {
            executor.submit(download_slide, row): row["file_name"]
            for _, row in batch.iterrows()
        }
        for future in futures:
            try:
                future.result()
            except Exception as e:
                print(f"  Download error: {e}")
    print("Downloads complete")

    # Process each slide
    for _, row in batch.iterrows():
        slide_name = row["file_name"]
        local_path = local_wsi / slide_name
        emb_path = drive_emb / slide_name.replace(".svs", ".h5")

        print(f"\n  Processing {slide_name}...")

        if not local_path.exists():
            print(f"  Download failed, skipping")
            continue

        result = process_slide(local_path, slide_name)

        if result is None:
            print(f"  No patches, skipping")
            local_path.unlink()
            continue

        features, coords = result

        # Save compressed HDF5 to Drive
        with h5py.File(emb_path, "w") as f:
            f.create_dataset("features", data=features,
                             compression="gzip", compression_opts=9)
            f.create_dataset("coords", data=coords,
                             compression="gzip", compression_opts=9)
            f.attrs["model"] = "uni"
            f.attrs["slide_name"] = slide_name
            f.attrs["num_patches"] = features.shape[0]
            f.attrs["feature_dim"] = features.shape[1]

        print(f"  Saved {features.shape} to Drive")

        # Delete processed WSI
        local_path.unlink()

    # Clean up any remaining files
    for f in local_wsi.glob("*.svs"):
        f.unlink()

    print(f"\nBatch complete, disk cleaned")

print(f"\nAll done! Embeddings saved to {drive_emb}")